# 02 – Signallabor: Gebäudespezifische Signalverarbeitung
**Netzdienlichkeit · PEXL · Experiment C**

Ziel: Dasselbe externe Signal an unterschiedliche thermische Gebäudeeigenschaften anpassen und die resultierenden Aktivierungsverläufe vergleichen. Dieses Notebook ist eigenständig ausführbar; es benötigt nicht die Ausführung von Notebook 01.

**Bedienung:** Entpacke das ZIP-Paket vollständig. Belasse `signallabor_data.csv` und `assets/` neben den Notebooks. Öffne das Notebook in Jupyter oder VS Code, wähle einen Python-Kernel und führe die Zellen von oben nach unten mit Shift+Enter aus. Benötigt werden pandas, numpy und matplotlib (`%pip install pandas numpy matplotlib`, falls nötig). Ändere nur die als **EXPERIMENT** gekennzeichneten Parameter und beantworte die Leitfragen in Markdown. Keine eigene Programmierung erforderlich.

Ein Signal $s(t)$ zwischen 0 und 1 beschreibt die **angeforderte zusätzliche Flexibilitätsnutzung**. 0 bedeutet keine zusätzliche Aktivierung (kein generelles Verbrauchsverbot), 1 maximale Aktivierung innerhalb der verfügbaren Flexibilität. Es ist weder eine direkte Leistungsanforderung in kW noch ein Nachweis tatsächlicher Lastverschiebung.

## Lernziele und Aufbau

Dieses Notebook dient zugleich als Skriptum und Übungsanleitung. Nach der Bearbeitung kannst du (1) die thermischen Kennwerte eines Gebäudes aus PEExcel erklären und übernehmen, (2) charakteristische Lade- und Entladezeiten berechnen, (3) externe Signale mit diesen Zeitmaßstäben verarbeiten und (4) die Grenzen der vereinfachten Methoden gegenüber der stündlichen PEExcel-Simulation begründen.

Arbeitsfolge: Ausgangssignal → Gebäudekennwerte in Excel → thermisches Modell → Signalverarbeitung → Gebäudevergleich → Export und Hypothese für PEExcel.

Einheitenkonvention: Alle spezifischen thermischen Größen beziehen sich auf die Nettogrundfläche (NGF; in PEExcel `NFA_total`, Nettogeschoßfläche).

![Vom externen Signal zur Gebäudereaktion](assets/signal_workflow.svg)


## 1. Ausgangsdaten
| Auswahl | Größe | Freigabe bei |
|---|---|---|
| wind | Onshore-Winderzeugung [MW; Einheit gemäß Generation-by-Type-Konvention] | hohen Werten |
| exaa | Day-Ahead EXAA [€/MWh] | niedrigen Werten |
| mc | Day-Ahead MC [€/MWh] | niedrigen Werten |
| co2 | Strommix-THG-Intensität (LCA) [g CO₂eq/kWh] | niedrigen Werten |
| load_error | Istlast minus Day-Ahead-Prognose [MW] | negativen Werten |

Datengrundlage: bereitgestellte ENTSO-E-Jahresblätter, `exaa.csv`, stündliche MC-Spalten aus `Day Ahead Preise_AT_20-24 - FlexSignale.xlsx`, Electricity Map 2021/2022 (LCA), `Day-ahead Forecast vs. Actual Load [MW].csv`. Die vorhandenen binären Spalten werden nicht verwendet.

**Zeitachse (Draft):** 2021 und 2022, jeweils 8760 Werte ohne Fehlwerte. Wind wird in ursprünglicher chronologischer Reihenfolge übernommen (inkl. doppelter lokaler Herbststunde, ohne leere Schlusszeile). Preis und Lastfehler liegen bereits als 8760-Stunden-Reihen vor; CO₂ liegt in UTC vor. Der gemeinsame Modellzeitstempel ist eine synthetische Stundenachse, keine nachgewiesene zeitzonenrichtige Synchronisation. Vergleiche hier nur Signalstatistiken; vor zeitgenauer gemeinsamer Simulation oder Kreuzkorrelation muss die Zeitbasis mit PEExcel abgeglichen werden. 2020 (Schaltjahr und abweichendes CO₂-Schema) sowie weitere Jahre sind in diesem Draft nicht enthalten.

**Interpretation:** Lastprognosefehler werden rückblickend ausgewertet; sie sind zum Day-Ahead-Zeitpunkt nicht bekannt. Niedrige Preise, niedrige THG-Intensität und hohe Windproduktion sind unterschiedliche Zielgrößen und kein Beleg für lokale Netzentlastung.

In [ ]:
# Nur bei fehlenden Paketen: Kommentarzeichen entfernen und Zelle ausführen.
# %pip install pandas numpy matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
%config InlineBackend.figure_formats = ['svg']

data = pd.read_csv(Path("signallabor_data.csv"), parse_dates=["model_timestamp"])
assert data.groupby("year").size().eq(8760).all()
assert not data.isna().any().any()
info = {
    "wind": ("Wind Onshore", "MW", "high"),
    "exaa": ("EXAA", "EUR/MWh", "low"),
    "mc": ("Market Coupling", "EUR/MWh", "low"),
    "co2": ("THG-Intensität LCA", "g CO₂eq/kWh", "low"),
    "load_error": ("Istlast − Prognose", "MW", "low"),
}
display(data.groupby("year").size().rename("Stunden"))

## 2. Ausgangssignal aus Notebook 01

Übertrage deine gewählten Parameter aus Notebook 01. Die Signalberechnung ist identisch. Bei Quantil-Ramps ist `share = (Vollaktivierungsanteil, Aktivierungsbeginn)`. Das Fenster enthält die aktuelle Stunde, frühere und spätere Stunden. Für `fixed` ist `threshold` bei kontinuierlicher Aktivierung `(keine, volle Aktivierung)` in Originaleinheiten.

Ein rollierendes Quantil definiert ein **lokales Referenzniveau**. `window=(Stunden davor, Stunden danach)` enthält auch die aktuelle Stunde; `(0,24)` umfasst also 25 Werte. Am Jahresrand verkürzt sich das Fenster auf verfügbare Stunden. Die Aktivierungsfunktion aus Notebook 01 setzt den Abstand zur Schwelle in eine dimensionslose Anforderung um. Die spätere zeitliche Verarbeitung in diesem Notebook ist ein separater Schritt.

```text
                       Vergangenheit | Zukunft
                                    t
(24, 0)       ███████████████████████│
(12, 12)                  ███████████│████████████
(0, 24)                              │████████████████████████
(6, 18)                        ██████│██████████████████
```

Vorausschauende Fenster auf historischen Messwerten setzen perfekte Voraussicht voraus. Bei Day-Ahead-Preisen können Werte innerhalb des veröffentlichten Horizonts bekannt sein.

![Zeitfenster für rollierende Quantile](assets/rolling_window.svg)


![Aktivierungsfunktionen als Wiederholung](assets/activation_functions.svg)

Im Monatsdiagramm steht jeder Punkt für eine zusammenhängende Phase. Grün markiert Stunden ab der Phasenschwelle `phase_cutoff`, Grau die übrigen Stunden. Die Dauer wird dem Monat des Phasenbeginns zugeordnet, auch wenn die Phase in den nächsten Monat reicht. Die blaue Linie zeigt das Monatsmittel aller stündlichen Aktivierungswerte auf der rechten Achse. Bei einem binären Signal entspricht dieses Mittel dem Anteil aktiver Stunden.

In den Boxplots markieren der Querstrich den Median und der weiße Kreis den Mittelwert der Phasendauer. Unter jedem Monat stehen beide Werte in Stunden (Mdn und Ø), getrennt nach Signal und Nichtsignal.


In [ ]:
def rolling_quantile(x, q, window):
    before, after = window
    if not all(isinstance(h, (int, np.integer)) and h >= 0 for h in window):
        raise ValueError("window: zwei nichtnegative ganze Stundenwerte")
    if not 1 <= before + after < len(x):
        raise ValueError("window: Summe der Stunden muss zwischen 1 und 8759 liegen")
    padded = pd.Series(np.r_[x.to_numpy(dtype=float), np.full(after, np.nan)])
    return pd.Series(padded.rolling(before + after + 1, min_periods=1)
                     .quantile(q).shift(-after).iloc[:len(x)].to_numpy(), index=x.index)


def reference(x, method, q, window):
    if method == "annual":
        return pd.Series(x.quantile(q), index=x.index)
    if method == "monthly":
        return x.groupby(x.index.month).transform(lambda a: a.quantile(q))
    if method == "rolling":
        return rolling_quantile(x, q, window)
    raise ValueError("method: fixed, annual, monthly oder rolling")


def activation_curve(a, activation):
    a = np.clip(a, 0, 1)
    if activation == "linear":
        return a
    if activation == "smoothstep":
        return a*a*(3-2*a)
    if activation == "sigmoid":
        # Rescaled logistic: exactly 0 and 1 at the ramp endpoints.
        f = lambda v: 1/(1+np.exp(-10*(v-.5)))
        return (f(a)-f(0))/(f(1)-f(0))
    raise ValueError("activation: binary, linear, sigmoid oder smoothstep")


def make_signal(year, source, method="annual", share=.30,
                window=(0,24), threshold=0, activation="binary"):
    x = data.loc[data.year.eq(year)].set_index("model_timestamp")[source].sort_index()
    if len(x) != 8760:
        raise ValueError("Modelljahr mit 8760 Stunden erforderlich")
    # Intern: höhere Werte sind stets günstiger.
    direction = 1 if info[source][2] == "high" else -1
    z = direction*x
    if activation == "binary":
        if method == "fixed":
            low = pd.Series(direction*float(threshold), index=x.index)
        else:
            if not 0 < share < 1:
                raise ValueError("share muss zwischen 0 und 1 liegen")
            low = reference(z, method, 1-share, window)
        high = low.copy()
        s = (z > low).astype(float)
    else:
        if method == "fixed":
            # threshold = (keine Aktivierung, volle Aktivierung), in Originaleinheit
            a, b = threshold
            if not direction*a < direction*b:
                raise ValueError("threshold: (keine, volle Aktivierung), Richtung beachten")
            low = pd.Series(direction*a, index=x.index)
            high = pd.Series(direction*b, index=x.index)
        else:
            full, start = share
            if not 0 < full < start < 1:
                raise ValueError("share: (Vollaktivierungsanteil, maximaler Aktivierungsanteil)")
            low = reference(z, method, 1-start, window)
            high = reference(z, method, 1-full, window)
        width = high-low
        # Degenerate quantiles: use a strict binary threshold where both coincide.
        a = ((z-low)/width.where(width>0)).where(width>0, (z>low).astype(float))
        s = activation_curve(a, activation)
    if source == "load_error":
        s = s.where(x < 0, 0)
    return x, direction*low, direction*high, s.astype(float).clip(0,1)


def periods(s, cutoff=.5):
    state = (s >= cutoff).astype(int)
    return state.groupby(state.ne(state.shift()).cumsum()).agg(state="first", hours="size")


def stats(s, cutoff=.5):
    r = periods(s, cutoff)
    on, off = r.loc[r.state.eq(1),"hours"], r.loc[r.state.eq(0),"hours"]
    return pd.Series({
        "Mittlerer Aktivierungsgrad [%]": 100*s.mean(),
        "Äquivalente Vollaktivierungsstunden [h]": s.sum(),
        "Stunden mit Aktivierung > 0 [h]": (s>0).sum(),
        "Stunden mit Aktivierung ≥ 0,9 [h]": (s>=.9).sum(),
        f"Stunden mit Aktivierung ≥ {cutoff:g} [h]": (s>=cutoff).sum(),
        f"Anteil Stunden ≥ {cutoff:g} [%]": 100*(s>=cutoff).mean(),
        "Aktivierungsphasen": len(on),
        "Mittlere Aktivierungsdauer [h]": on.mean(),
        "Median Aktivierungsdauer [h]": on.median(),
        "Mittlere Nichtaktivierungsdauer [h]": off.mean(),
        "Median Nichtaktivierungsdauer [h]": off.median(),
        "Längste Nichtaktivierung [h]": off.max(),
        "Mittlere absolute Änderung je Stunde": s.diff().abs().mean(),
    })


def detail(s, interval):
    result = s.loc[interval[0]:interval[1]]
    if result.empty:
        raise ValueError("Der Zeitraum liegt außerhalb des Modelljahres.")
    return result


def fill_activation(ax, signals, interval):
    for name, signal in signals.items():
        values = detail(signal, interval)
        # Each value applies for a full hour, including the last displayed value.
        edges = values.index.insert(len(values), values.index[-1] + pd.Timedelta(hours=1))
        ax.fill_between(edges, 0, np.r_[values.to_numpy(), values.iloc[-1]],
                        step="post", alpha=.24, linewidth=0, label=name,
                        **({"color": "tab:green"} if len(signals) == 1 else {}))
    ax.set(ylim=(0, 1), ylabel="Aktivierung [0–1]")


def plot_signal(x, low, high, signals, source, interval):
    d = detail(x, interval)
    name, unit, direction = info[source]
    favorable, inactive = ("darüber", "darunter") if direction == "high" else ("darunter", "darüber")
    fig, ax = plt.subplots(figsize=(12, 4))
    activity = ax.twinx()
    fill_activation(activity, signals, interval)
    ax.set_zorder(activity.get_zorder() + 1)
    ax.patch.set_visible(False)

    ax.plot(d.index, d, color="tab:blue", linewidth=1, label=name)
    if low.equals(high):
        label = f"Schwelle ({favorable} aktiv, {inactive} und darauf inaktiv)"
    else:
        label = f"Aktivierungsbeginn ({favorable} zunehmend, {inactive} und darauf inaktiv)"
        ax.plot(d.index, high.loc[d.index], ":", color="tab:red", linewidth=1,
                label=f"Vollaktivierung ({favorable} und darauf voll aktiv)")
    ax.plot(d.index, low.loc[d.index], "--", color="tab:orange", linewidth=1, label=label)
    if source == "load_error":
        ax.axhline(0, color="0.4", linestyle="-.", linewidth=1,
                   label="Zusatzgrenze 0 MW (darüber und darauf inaktiv)")

    ax.set_ylabel(f"{name} [{unit}]")
    handles, labels = ax.get_legend_handles_labels()
    area_handles, area_labels = activity.get_legend_handles_labels()
    ax.legend(handles + area_handles, labels + area_labels, loc="upper left")
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()


def plot_monthly_phases(s, cutoff=.5):
    from matplotlib.ticker import LogLocator, NullLocator, StrMethodFormatter

    runs = periods(s, cutoff)
    start_positions = runs["hours"].cumsum() - runs["hours"]
    runs["month"] = s.index[start_positions.to_numpy()].month
    monthly_mean = 100 * s.groupby(s.index.month).mean()

    fig, ax = plt.subplots(figsize=(13, 5))
    rng = np.random.default_rng(0)
    cutoff_label = f"{cutoff:g}".replace(".", ",")
    groups = [
        (1, -.19, "tab:green", f"Signal (≥ {cutoff_label})"),
        (0, .19, "0.45", f"Nichtsignal (< {cutoff_label})"),
    ]
    for row, (state, offset, color, label) in enumerate(groups):
        by_month = runs.loc[runs["state"].eq(state)].groupby("month")["hours"]
        for month, hours in by_month:
            position = month + offset
            ax.boxplot(
                hours, positions=[position], widths=.28, manage_ticks=False,
                patch_artist=True, showfliers=False, showmeans=True,
                boxprops={"facecolor": color, "edgecolor": color, "alpha": .16},
                whiskerprops={"color": color, "alpha": .75},
                capprops={"color": color, "alpha": .75},
                medianprops={"color": color, "linewidth": 1.6, "zorder": 4},
                meanprops={"marker": "o", "markersize": 4.5, "markerfacecolor": "white",
                           "markeredgecolor": color, "zorder": 5},
            )
            median = f"{hours.median():g}".replace(".", ",")
            mean = f"{hours.mean():.1f}".replace(".", ",")
            ax.text(month, -.10 - .057 * row, f"Mdn {median} · Ø {mean}",
                    transform=ax.get_xaxis_transform(), ha="center", va="top",
                    color=color, fontsize=8, clip_on=False)
            jitter = rng.uniform(-.13, .13, size=len(hours))
            ax.scatter(position + jitter, hours, color=color, s=10, alpha=.65,
                       linewidths=0, zorder=3)
        ax.scatter([], [], color=color, s=18, label=label)

    months = "Jan Feb Mär Apr Mai Jun Jul Aug Sep Okt Nov Dez".split()
    ax.set_xticks(range(1, 13), months)
    ax.set(xlim=(.5, 12.5), yscale="log", ylim=(.8, None),
           ylabel="Phasendauer [h, logarithmisch]")
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.grid(axis="y", alpha=.2)
    ax.set_axisbelow(True)

    mean_axis = ax.twinx()
    mean_axis.plot(monthly_mean.index, monthly_mean, color="tab:blue", linewidth=1.2,
                   marker="D", markersize=4, label="Mittlere Aktivierung")
    mean_axis.set(ylim=(0, 100), ylabel="Mittlere Aktivierung [%]")
    mean_axis.tick_params(axis="y", colors="tab:blue")
    mean_axis.yaxis.label.set_color("tab:blue")

    phase_handles, phase_labels = ax.get_legend_handles_labels()
    mean_handles, mean_labels = mean_axis.get_legend_handles_labels()
    mean_axis.legend(phase_handles + mean_handles, phase_labels + mean_labels,
                     ncol=3, loc="lower center", bbox_to_anchor=(.5, 1))
    fig.tight_layout()
    plt.show()


def visualize(x, low, high, s, source, interval, cutoff=.5):
    plot_signal(x, low, high, {"Aktivierung": s}, source, interval)
    fig, ax = plt.subplots(figsize=(12,3))
    im = ax.imshow(s.to_numpy().reshape(365,24).T,aspect="auto",origin="lower",vmin=0,vmax=1,cmap="Greens",extent=(.5,365.5,-.5,23.5))
    ax.set(xlabel="Tag im Modelljahr",ylabel="Stunde",title="Aktivierung im Jahresverlauf")
    fig.colorbar(im,ax=ax); plt.show()
    plot_monthly_phases(s, cutoff)


def compare(signals, interval, cutoff=.5):
    display(pd.DataFrame({name:stats(s,cutoff) for name,s in signals.items()}).round(2))
    df=pd.DataFrame(signals)
    fig,ax=plt.subplots(figsize=(12,4))
    fill_activation(ax, signals, interval)
    ax.set_title("Detailvergleich")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()
    fig,ax=plt.subplots(figsize=(10,3))
    (100*df.groupby(df.index.month).mean()).plot.bar(ax=ax)
    ax.set(xlabel="Monat",ylabel="Mittlere Aktivierung [%]",ylim=(0,100)); plt.show()
    fig,ax=plt.subplots(figsize=(10,3))
    for name,s in signals.items():
        ax.hist(s,bins=np.linspace(0,1,21),histtype="step",linewidth=1.7,label=name)
    ax.set(xlabel="Aktivierung",ylabel="Stunden",title="Verteilung der Aktivierungsgrade"); ax.legend(); plt.show()


def export_signal(s, year, name, parameters, prefix):
    assert len(s)==8760 and s.notna().all() and s.between(0,1).all()
    folder=Path("output"); folder.mkdir(exist_ok=True)
    safe="".join(c if c.isalnum() else "_" for c in name)
    stem=folder/f"{prefix}_{year}_{safe}"
    s.to_csv(f"{stem}.csv",index=False,header=False,float_format="%.8f")
    pd.DataFrame({"hour":range(1,8761),"model_timestamp":s.index,"signal":s.to_numpy()}).to_csv(f"{stem}_documented.csv",index=False)
    pd.DataFrame([dict(parameters,year=year,name=name)]).to_csv(f"{stem}_parameters.csv",index=False)
    print(f"{stem}.csv – 8760 Stunden")

In [ ]:
# EXPERIMENT C – Ausgangssignal
year = 2022
source = "co2"
method = "rolling"
window = (0,24)
activation = "linear"
share = (.20,.50)  # bei binary eine Zahl, z. B. .30
threshold = (250,100)  # bei binary eine Zahl
interval = ("2022-01-15", "2022-01-31")
phase_cutoff = .5
x,low,high,base = make_signal(year,source,method,share,window,threshold,activation)
display(stats(base,phase_cutoff).round(2).to_frame("Ausgangssignal"))
plot_monthly_phases(base, phase_cutoff)


## 3. Thermische Gebäudecharakteristik: Theorie und PEExcel

### 3.1 Wärmeverlustleitwerte und Wärmekapazität

Die österreichische bauphysikalische Nomenklatur unterscheidet den Transmissionsleitwert $L_T$ und den Lüftungsleitwert $L_V$. Für das vereinfachte Modell gilt $L=L_T+L_V$. Bei einer Temperaturdifferenz $T_i-T_a$ ergibt sich die Wärmeverlustleistung $P_{\mathrm{loss}}=L(T_i-T_a)$.

Für den Vergleich unterschiedlich großer Gebäude beziehen wir alle extensiven Größen auf die NGF $A_{\mathrm{NGF}}$:

$$L'_T=\frac{L_T}{A_{\mathrm{NGF}}},\quad L'_V=\frac{L_V}{A_{\mathrm{NGF}}},\quad L'=L'_T+L'_V,\quad C'=\frac{C}{A_{\mathrm{NGF}}},\quad P'_{\max}=\frac{P_{\max}}{A_{\mathrm{NGF}}}.$$

| Symbol | Bedeutung | Absolut | Spezifisch pro m² NGF |
|---|---|---|---|
| $L_T$, $L'_T$ | Transmissionsleitwert | W/K | W/(m²K) |
| $L_V$, $L'_V$ | Lüftungsleitwert | W/K | W/(m²K) |
| $L$, $L'$ | Gesamtwärmeverlustleitwert | W/K | W/(m²K) |
| $C$, $C'$ | Wirksame thermische Wärmekapazität | Wh/K | Wh/(m²K) |
| $P_{\max}$, $P'_{\max}$ | Max. raumseitige thermische Heizleistung | W | W/m² |
| $\tau$ | Gebäudezeitkonstante | h | h (unverändert) |

### 3.2 Wo stehen die Größen in PEExcel?

Öffne die maßgebliche Variante im PEExcel. Suche im Blatt **IN** bzw. **OUT** nach dem angegebenen `var_name` oder der deutschen Bezeichnung. **SIM** enthält stündliche Werte. Die folgenden OUT-Zahlen stammen aus dem bereitgestellten Beispiel; deine eigene Variante kann davon abweichen.

| Größe / Kennwert | Blatt | `var_name` / Bezeichnung | Einheit | Beispiel |
|---|---|---|---|---:|
| NGF | IN | `NFA_total` – NGF Quartier | m² | nicht angegeben |
| $C'$ | IN | `heat_capacity_effective_m2` – Spezif. wirksame Wärmekapazität | Wh/(m²K) | nicht direkt angegeben |
| $L_T$ | OUT | `transmittance` | W/K | 3023,2 |
| $L_V$ | OUT | `ventilation` | W/K | 4036,0 |
| $L$ | OUT | `heat_loss` | W/K | 7059,2 |
| $L'_T$ | IN | `transmittance_Wm2`; auch OUT-Zeile „Spez. Transmissionsleitwert L'T“ | W/(m²K) | 0,2560 |
| $L'_V$ | OUT | `ventilation_Wm2` | W/(m²K) | 0,3418 |
| $L'$ | OUT | `heat_loss_Wm2` | W/(m²K) | 0,5978 |
| $\tau$ | OUT | `thermal_time_constant_h` | h | 341,28 |
| $P'_{\mathrm{Abgabe}}$ | IN | `QHmax_room_m2` – spez. thermische Leistung Abgabesystem | W/m² | nicht angegeben |
| $P'_{\mathrm{Erzeugung}}$ | IN | `QHmax_1el_m2`, `QHmax_2th_m2`, `QHmax_3el_m2`, `QHmax_4th_m2` (Summe) | W/m² | nicht angegeben |
| $P'_{\max}$ | IN | `QHmax_total_m2` – maximal verfügbare thermische Heizleistung | W/m² | nicht angegeben |
| $T_{\min}$ | IN | `Tsetheat_min` | °C | nicht angegeben |
| $T_{\max}$ | IN | `Tsetheat_flex` | °C | nicht angegeben |
| $T_a$ | SIM | `Ta` – stündliche Außentemperatur | °C | Zeitreihe |
| Jahresmittel $T_a$ | OUT | `Ta_annual_avg` | °C | 11,06 |
| Heizperiodenmittel $T_a$ | OUT | `Ta_avg_heating_period` | °C | 4,93 |
| Kühlperiodenmittel $T_a$ | OUT | `Ta_avg_cooling_period` | °C | 18,94 |
| Modell-Heizgradtage | OUT | `heating_degree_days` | Kd | 3106 |
| HGT 20/12 | OUT | `Heating_degree_days_20_12` | Kd | 2929 |
| Mittlere Entladedauer | OUT | „Mittlere Entladedauer“ (var_name nicht angegeben) | h | 28,75 |

Hinweis zur Schreibweise: In einem Schemaexport können `heat_capacity_effective_m²` und `QHmax_room_m²` mit hochgestellter 2 erscheinen. Die SIM-Formeln verwenden die ASCII-Namen mit `m2`. Verwende beim automatisierten Zugriff den Namen der jeweiligen Version.

### 3.3 Wie entsteht der Lüftungsleitwert?

Transmission ist im Modell im Wesentlichen über Bauteilflächen, U-Werte und Wärmebrücken definiert. Lüftung hängt zusätzlich von Nutzungsprofilen, Infiltration, Fensterlüftung und Wärmerückgewinnung ab. Aus dem effektiven Volumenstrom $\dot V_{\mathrm{eff}}$ [m³/h] folgt $L_V(t)=c_{\mathrm{air}}\dot V_{\mathrm{eff}}(t)$ mit der volumetrischen Wärmekapazität der Luft (ca. 0,34 Wh/(m³K)).

PEExcel summiert die Lüftungswärmeverluste der Heizperiode in OUT:

```excel
=QV_inf_winter+QV_window_winter+QV_mechvent_winter
```

Der für diese Übung verwendete effektive Leitwert wird näherungsweise rückgerechnet (QV_winter in kWh/m² NGF):

$$L'_V\approx\frac{1000\,|QV_{\mathrm{winter}}|}{24\,HGT_{\mathrm{Modell}}}.$$

```excel
=ABS(QV_winter)*1000/(heating_degree_days*24)
```

Das ist ein temperaturdifferenzgewichteter Näherungswert für die Heizperiode. Zähler und Nenner müssen denselben Zeitraum, dieselbe Vorzeichenkonvention und denselben Temperaturbezug verwenden. Die dynamischen Innenraumtemperaturen bzw. gegebenenfalls Wärmegewinne durch Lüftung können zu Abweichungen führen. Der berechnete Wert ist daher kein konstanter physikalischer Luftwechsel.

**Heizgradtage:** Im Modell werden positive stündliche Differenzen zur Mindest-Heiztemperatur innerhalb der definierten Heizperiode summiert und durch 24 geteilt:

$$HGT_{\mathrm{Modell}}=\frac{1}{24}\sum_{t\in\mathrm{Heizperiode}}\max(T_{\min}-T_a(t),0).$$

```excel
=SUMMENPRODUKT(sim[is_heating_period];(sim[Ta]<Tsetheat_min);Tsetheat_min-sim[Ta])/24
```

Die HGT 20/12 verwenden dagegen die Tagesmitteltemperatur und eine feste Heizgrenze von 12 °C bei 20 °C Bezugsraumtemperatur. Sie sind eine Klimakenngröße. Für die Rückrechnung des Lüftungsleitwerts verwende die modellspezifischen HGT (im Beispiel 3106 Kd statt 2929 Kd), sofern die Abgrenzung zu den Wärmeverlusten passt.

### 3.4 Ein-Knoten-Modell und Gebäudezeitkonstante

Für konstante Außentemperatur und konstantes $L'$ gilt im Heizfall ohne solare und interne Gewinne die spezifische Wärmebilanz:

$$C'\frac{dT_i}{dt}=P'_{\mathrm{heat}}-L'(T_i-T_a).$$

Daraus ergibt sich die Zeitkonstante (NGF kürzt sich heraus):

$$\boxed{\tau=\frac{C'}{L'}=\frac{C'}{L'_T+L'_V}}.$$

```excel
=heat_capacity_effective_m2/heat_loss_Wm2
```

Nach einer Zeitkonstante bleiben beim unbeheizten Gebäude 36,8 % der ursprünglichen Temperaturdifferenz zur Außenluft erhalten. Die Zeitkonstante ist ein Maß der thermischen Trägheit, keine unmittelbar nutzbare Dauer des Heizungsstillstands.

![Thermisches Ein-Knoten-Modell](assets/thermal_rc_model.svg)


![Exponentieller Temperaturabfall nach einer Zeitkonstante](assets/thermal_time_constant.svg)


### 3.5 Thermische Speicherkapazität und Entladedauer

Das nutzbare Komfortband ist $\Delta T=T_{\max}-T_{\min}$. Die sensible spezifische Speicherkapazität beträgt näherungsweise

$$E'_{\mathrm{th}}=C'\Delta T \quad [\mathrm{Wh/m^2}].$$

Für $P'_{\mathrm{heat}}=0$ gilt bei $T_a<T_{\min}$:

$$T_i(t)=T_a+(T_{\max}-T_a)e^{-t/\tau}.$$

Auflösen nach dem Zeitpunkt, an dem $T_i=T_{\min}$ erreicht wird, ergibt

$$\boxed{t_d=\tau\ln\left(\frac{T_{\max}-T_a}{T_{\min}-T_a}\right)}.$$

```excel
=thermal_time_constant_h*LN((Tsetheat_flex-Ta_avg_heating_period)/(Tsetheat_min-Ta_avg_heating_period))
```

Die Formel illustriert die mittlere Entladedauer bei Heizperiodenmitteltemperatur. Sie ist im Allgemeinen deutlich kleiner als $\tau$. Ein größerer Temperaturhub oder wärmere Außenluft ermöglicht längeres Überbrücken. Der Wert bei mittlerer Temperatur ist nicht der Mittelwert aller stündlichen Entladezeiten.

### 3.6 Maximale Heizleistung und Ladedauer

Die maximal verfügbare raumseitige thermische Heizleistung ist durch die Wärmeabgabe und die Summe der vier Wärmeerzeugerleistungen begrenzt:

$$P'_{\max}=\min\left(P'_{\mathrm{Abgabe}},\sum_{i=1}^{4} P'_{\mathrm{Erzeugung},i}\right).$$

Diese Größe ist in PEExcel in **IN** als `QHmax_total_m2` angelegt und wird direkt ins Experiment übernommen. Die Excel-Formel lautet:

```excel
=MIN(QHmax_room_m2;SUMME(QHmax_1el_m2;QHmax_2th_m2;QHmax_3el_m2;QHmax_4th_m2))
```

Dies ist eine theoretische Obergrenze für die gesamte Heizleistung, nicht allein für zusätzliche flexible Heizleistung. In SIM ist `Qh_to_room_min` der Mindestheizbedarf, der durch Erzeugerprioritäten gedeckt wird. Flexibel verfügbare Erzeugerleistung (System 1 oder 3, soweit aktiviert) ist auf die verbleibende Leistung nach Deckung des Mindestbedarfs begrenzt. Weiter begrenzen die elektrische Flexgrenze `FLEX_grid_maxpower_Wm2`, Signal, COP, konkurrierende flexible Verbraucher und raumseitiger Bedarf die tatsächliche Beladung. Bei Wärmepumpen sind elektrische und thermische Leistung zu unterscheiden. Die analytische Ladezeit geht vom idealisierten Betrieb bei maximaler gesamter thermischer Leistung aus; bei gleicher übriger Randbedingung ist sie eine untere Schranke für eine leistungsbegrenzte Beladung.

Bei konstanter maximaler thermischer Heizleistung ist die Gleichgewichtstemperatur

$$T_{\mathrm{eq}}=T_a+\frac{P'_{\max}}{L'}.$$

Die Raumtemperatur steigt ausgehend von $T_{\min}$ gemäß

$$T_i(t)=T_{\mathrm{eq}}+(T_{\min}-T_{\mathrm{eq}})e^{-t/\tau}.$$

Für $T_{\mathrm{eq}}>T_{\max}$ ergibt sich die Ladezeit

$$\boxed{t_c=\tau\ln\left(\frac{T_{\mathrm{eq}}-T_{\min}}{T_{\mathrm{eq}}-T_{\max}}\right)}.$$

Ist $T_{\mathrm{eq}}\le T_{\max}$, wird das Komfortband unter diesen Annahmen nicht vollständig geladen. Bei der Ladedauer wird laufender Wärmeverlust mitberücksichtigt; sie entspricht daher nicht einfach $E'_{\mathrm{th}}/P'_{\max}$.

![Thermisches Laden und Entladen](assets/thermal_charging_discharging.svg)

*Beispiel mit C′ = 204 Wh/(m²K), L′ = 0,5978 W/(m²K), P′max = 40 W/m², Ta = 4,93 °C und Komfortband 20–22 °C. Die Heizleistung und Temperaturen sind illustrative Annahmen.*


### 3.7 Theorie vs. PEExcel

| Vereinfachtes Modell im Notebook | Stündliche PEExcel-Simulation |
|---|---|
| Ein Temperaturknoten, konstantes $C'$ | Gekühlte und ungekühlte Bereiche mit thermischen Zuständen |
| Konstantes $L'$ | Stündliche Lüftungsprofile und WRG, Transmission |
| Konstantes $T_a$ | Stündliches Wetter (`sim[Ta]`) |
| Keine inneren/solaren Gewinne | Wärmegewinne werden bilanziert |
| Konstantes $P'_{\max}$ bzw. null | Heizbedarf, Erzeugerprioritäten, COP und verfügbare Leistung |
| Vollständiges Laden/Entladen | Tatsächliche Raumtemperatur, Flexpotenzial und Regelung je Stunde |

Die analytischen Zeiten sind charakteristische Zeitmaßstäbe zur Signalgestaltung. Sie sind keine Vorhersage der tatsächlich erreichten Raumtemperatur oder Lastverschiebung.

In [ ]:
# EXPERIMENT C1 – Spezifische Gebäudekennwerte aus PEExcel übernehmen
# Beispiel-OUT: L_T'=0.2560, L_V'=0.3418, tau=341.28 h.
# C'=tau*L' ergibt rund 204 Wh/(m²K); im eigenen Projekt direkt aus IN übernehmen.
# Heizleistungen und Komforttemperaturen sind hier illustrative Annahmen.
building = dict(
    C_spec=204.0,          # Wh/(m²NGF K) – heat_capacity_effective_m2
    L_T_spec=0.2560,       # W/(m²NGF K) – transmittance_Wm2
    L_V_spec=0.3418,       # W/(m²NGF K) – ventilation_Wm2
    P_max_spec=40.0,       # W/m²NGF thermisch – QHmax_total_m2 (Beispiel)
    T_out=4.93,            # °C – Ta_avg_heating_period
    T_min=20.0,            # °C – Tsetheat_min (Beispiel)
    comfort_band=2.0,      # K – Tsetheat_flex - Tsetheat_min (Beispiel)
)


def building_times(C_spec, L_T_spec, L_V_spec, P_max_spec,
                   T_out, T_min, comfort_band):
    if (C_spec <= 0 or min(L_T_spec, L_V_spec) < 0 or
        L_T_spec + L_V_spec <= 0 or min(P_max_spec, comfort_band) <= 0
        or T_out >= T_min):
        raise ValueError("Positive thermische Größen, nichtnegative Leitwerte und T_out < T_min erforderlich")
    L_spec = L_T_spec + L_V_spec
    tau = C_spec / L_spec
    T_max = T_min + comfort_band
    T_eq = T_out + P_max_spec / L_spec
    if T_eq <= T_max:
        raise ValueError("Heizleistung reicht nicht zur vollständigen Beladung; Leistung, Wetter oder Komfortband prüfen")
    t_discharge = tau * np.log((T_max-T_out)/(T_min-T_out))
    t_charge = tau * np.log((T_eq-T_min)/(T_eq-T_max))
    return dict(L_spec=L_spec, P_max_spec=P_max_spec, tau=tau,
                charging_time=t_charge, discharging_time=t_discharge,
                capacity=C_spec*comfort_band, T_max=T_max, T_eq=T_eq)


b = building_times(**building)
display(pd.Series({
    "Spez. Transmissionsleitwert L'T [W/(m²K)]": building["L_T_spec"],
    "Spez. Lüftungsleitwert L'V [W/(m²K)]": building["L_V_spec"],
    "Spez. Gesamtwärmeverlustleitwert L' [W/(m²K)]": b["L_spec"],
    "Max. thermische Heizleistung P'max [W/m²]": b["P_max_spec"],
    "Zeitkonstante [h]": b["tau"],
    "Ladedauer [h]": b["charging_time"],
    "Entladedauer [h]": b["discharging_time"],
    "Speicherkapazität [Wh/m² NGF]": b["capacity"],
    "Speicherkapazität [kWh/m² NGF]": b["capacity"]/1000,
}).round(3).to_frame("Gebäude"))

fig, ax = plt.subplots(figsize=(10,4))
tc = np.linspace(0,b["charging_time"],100)
td = np.linspace(0,b["discharging_time"],100)
ax.plot(tc,b["T_eq"]+(building["T_min"]-b["T_eq"])*np.exp(-tc/b["tau"]),label="Laden bei P'max")
ax.plot(b["charging_time"]+td,building["T_out"]+(b["T_max"]-building["T_out"])*np.exp(-td/b["tau"]),label="Entladen ohne Heizung")
ax.axhspan(building["T_min"],b["T_max"],alpha=.1,color="tab:green",label="Komfortband")
ax.axvline(b["charging_time"],linestyle="--",color="grey",alpha=.5)
ax.set(xlabel="Zeit [h]",ylabel="Temperatur [°C]",title="Idealisiertes Laden und Entladen")
ax.legend(); ax.grid(alpha=.2); plt.show()


### Selbstkontrolle – Kennwerte und Zeitmaßstäbe

1. Übertrage die Werte deiner PEExcel-Variante in die Parameterzelle. Vergleiche die berechnete Zeitkonstante mit OUT. Prüfe Einheiten und NGF-Bezug.
2. Vergleiche die berechnete Entladedauer mit „Mittlere Entladedauer“ in OUT. Im bereitgestellten Beispiel sind es 28,75 h; die zugrunde liegenden Komforttemperaturen wurden nicht angegeben. Der Default in diesem Notebook ist daher kein exakter Reproduktionsfall.
3. Ändere ausschließlich $T_a$ (z. B. −10, 0, 4,93 und 10 °C). Welche Kennwerte ändern sich? Welche bleiben konstant?
4. Verändere die maximale Heizleistung und erkläre, wann die vollständige Beladung nicht mehr möglich ist.

**Antwort:** …

## 4. Zeitliche Signalverarbeitung

| Methode | Prinzip | Bezug zum Gebäude |
|---|---|---|
| `base` | Ursprüngliche Aktivierung | Referenz |
| `rolling_mean` | Rückblickender gleitender Mittelwert | Mittelungsdauer, hier aus Ladedauer abgeleitet |
| `exponential` | Exponentieller Tiefpass | Filterzeitkonstante, hier aus Entladedauer abgeleitet |
| `ramp_rate` | Begrenzte stündliche Signaländerung | Hoch- und Herunterfahren in Lade- bzw. Entladedauer |
| `lookahead` | Aktuelle Führungsgröße gegen gewichtete zukünftige Werte vergleichen | Horizont aus Entladedauer, Gewichtung nach Gebäudezeitkonstante |

Die Filterparameter sind **heuristische Zuordnungen**. Sie folgen nicht zwingend aus der Gebäudephysik: eine Glättung kann günstige Stunden verspätet aktivieren; eine Ramp-Rate ist keine physikalische Temperatur- oder Leistungsgrenze. Die Werte können im Experiment verändert werden.

Die Schwellen im gemeinsamen Zeitdiagramm beziehen sich auf das Ausgangssignal (`base`). Die farbigen Flächen zeigen die Aktivierung nach der jeweiligen Verarbeitung.

**Exponentieller Filter:** $s_t=s_{t-1}+\alpha(u_t-s_{t-1})$, mit $\alpha=1-e^{-1\mathrm{h}/\tau_f}$. Dabei ist $u_t$ das Ausgangssignal und $\tau_f$ die Filterzeitkonstante.

**Zeitliche Ramp:** $s_t=s_{t-1}+\operatorname{clip}(u_t-s_{t-1},-1\mathrm{h}/t_d,1\mathrm{h}/t_c)$. Dies begrenzt die Aktivierungsänderung je Stunde, nicht die Ladeleistung selbst.

**Look-ahead:** Für die günstig orientierte Führungsgröße $z$ vergleichen wir den aktuellen Stundenwert mit dem zukünftigen gewichteten Mittel. Für $k=1,\ldots,N$ gilt $w_k=e^{-k\mathrm{h}/\tau}$ und

$$v(t)=z(t)-\frac{\sum_{k=1}^{N}w_k z(t+k)}{\sum_{k=1}^{N}w_k}.$$

Je größer der Vorteil $v(t)$, desto stärker die Aktivierung. Die Umrechnung erfolgt hier über eine lineare Ramp mit zwei absoluten Vorteilsschwellen in der Originaleinheit. Negative oder kleine Vorteile sollen keine zusätzliche Aktivierung auslösen. Das letzte Intervall ohne Zukunftswerte fällt auf das Ausgangssignal zurück; davor werden nur verfügbare Zukunftsstunden verwendet.

**Wichtig:** Look-ahead nutzt bei historischen Daten perfekte Voraussicht. Reale Anwendung erfordert verfügbare Prognosen. Die Regel betrachtet weder tatsächliche Raumtemperatur noch SOC noch Bedarf oder Leistungsgrenzen und ist keine MPC. Bei Lastprognosefehlern ist auch der aktuelle Fehler erst nach Beobachtung bekannt; Look-ahead darauf wäre rein hypothetisch.

**Zusammenhang zur Regelung:** Eine Signalglättung oder Ramp-Rate begrenzt zunächst nur die Aktivierungsanforderung. Sie erzwingt keine bestimmte Raumtemperatur. Das Look-ahead-Verfahren bildet ein relatives Vorteilssignal; es optimiert weder Speicherzustand noch Betriebskosten. Die Wahl der Lade- bzw. Entladedauer als Filterhorizont ist eine zu untersuchende Hypothese. Auch ein kurzer günstiger Zeitraum kann bei bereits teilweise geladenem Gebäude sinnvoll sein. Die Phasendauer-Auswertung mit `phase_cutoff` vereinfacht kontinuierliche Signale zu zwei Zuständen; sie ist keine tatsächliche Ein-/Ausschaltdauer der Wärmepumpe.


![Vergleich zeitlicher Filter](assets/temporal_filters.svg)

*Synthetischer Verlauf; illustriert Glättung, Verzögerung und Begrenzung der Änderungsgeschwindigkeit.*


![Vorausschauende Aktivierung vor Hochpreisphasen](assets/predictive_activation.svg)

*Schematisches Preisbeispiel mit 12 h Vorschau. Kein simulierter Speicherzustand.*


In [ ]:
def process(base, x, source, kind, mean_hours, filter_tau, charging_time,
            discharging_time, horizon, weight_tau, advantage_threshold):
    if kind=="base":
        s=base.copy()
    elif kind=="rolling_mean":
        s=base.rolling(mean_hours,min_periods=1).mean()
    elif kind=="exponential":
        s=base.ewm(alpha=1-np.exp(-1/filter_tau),adjust=False).mean()
    elif kind=="ramp_rate":
        values=base.to_numpy(); out=np.empty(len(values)); out[0]=values[0]
        for i in range(1,len(out)):
            out[i]=out[i-1]+np.clip(values[i]-out[i-1],-1/discharging_time,1/charging_time)
        s=pd.Series(out,index=base.index)
    elif kind=="lookahead":
        direction=1 if info[source][2]=="high" else -1
        z=direction*x
        future=pd.concat([z.shift(-k) for k in range(1,horizon+1)],axis=1)
        weights=np.exp(-np.arange(1,horizon+1)/weight_tau)
        future_mean=future.mul(weights,axis=1).sum(axis=1)/future.notna().mul(weights,axis=1).sum(axis=1)
        advantage=z-future_mean
        a,b=advantage_threshold
        if not 0<=a<b: raise ValueError("Vorteilsschwellen: 0 <= keine < volle Aktivierung")
        s=((advantage-a)/(b-a)).clip(0,1).fillna(base)
    else:
        raise ValueError(kind)
    if source=="load_error":
        # Preserve source's negative-error restriction even after smoothing.
        s=s.where(x<0,0)
    return s.clip(0,1)


def filter_params(b, advantage_threshold, mean_factor=1, filter_factor=1, horizon_factor=1):
    return dict(mean_hours=max(1,round(b["charging_time"]*mean_factor)),
                filter_tau=b["discharging_time"]*filter_factor,
                charging_time=b["charging_time"],discharging_time=b["discharging_time"],
                horizon=max(1,round(b["discharging_time"]*horizon_factor)),
                weight_tau=b["tau"],advantage_threshold=advantage_threshold)

In [ ]:
# EXPERIMENT C – zeitliche Verarbeitung
# Schwellen in Originaleinheit der Quelle; hier g CO₂eq/kWh Vorteil
advantage_threshold = (0,30)
mean_factor = 1.0  # Faktor auf Ladedauer
filter_factor = 1.0  # Faktor auf Entladedauer
horizon_factor = 1.0  # Faktor auf Entladedauer
kinds=["base","rolling_mean","exponential","ramp_rate","lookahead"]
params=filter_params(b,advantage_threshold,mean_factor,filter_factor,horizon_factor)
display(pd.Series(params,dtype=object).to_frame("Filterparameter"))
signals_C={kind:process(base,x,source,kind,**params) for kind in kinds}
compare(signals_C,interval,phase_cutoff)

### Auswertung C1 – hier antworten
1. Welche Operationen reduzieren kurze Aktivierungsspitzen? Bei welchen tritt eine Verzögerung auf?
2. Welche Varianten aktivieren während günstiger Stunden, welche auch danach? Vergleiche mit der ursprünglichen Führungsgröße.
3. Welche Phasendauern sind länger als die Ladedauer oder länger als die Entladedauer? Welche Konsequenzen könnten sich daraus ergeben?
4. Sind die äquivalenten Vollaktivierungsstunden ähnlich? Warum ist ein Vergleich der tatsächlich verschobenen Energie erst mit Gebäudesimulation möglich?

**Antwort:** …

In [ ]:
# Vergleich mit Führungsgröße und Zeitmaßstäben
plot_signal(x, low, high, signals_C, source, interval)
rows={}
for name,s in signals_C.items():
    r=periods(s,phase_cutoff)
    on=r.loc[r.state.eq(1),"hours"]; off=r.loc[r.state.eq(0),"hours"]
    rows[name]={"Aktivierungsphasen ≥ Ladedauer [%]":100*(on>=b["charging_time"]).mean(),
                "Nichtaktivierungsphasen > Entladedauer [%]":100*(off>b["discharging_time"]).mean()}
display(pd.DataFrame(rows).round(1))

## 5. Zwei Gebäude im Vergleich

Verändere die Gebäudeeigenschaften, halte das externe Signal konstant. Standardmäßig vergleichen wir ein Gebäude mit geringer und eines mit höherer wirksamer Speichermasse bei gleicher Hülle und Heizleistung. So lässt sich der Einfluss von $C'$ isolieren. Du kannst anschließend auch Hülle, Komfortband oder Leistung verändern.

Die folgenden zwei Varianten sind illustrative spezifische Kennwerte, keine klassifizierten Referenzgebäude. Übernimm anschließend die tatsächlichen Kennwerte zweier PEExcel-Varianten. Die Leistungen und Temperaturen werden zum Isolieren des Einflusses der Speichermasse zunächst konstant gehalten.

In [ ]:
# EXPERIMENT C2 – Gebäudevarianten
buildings = {
    "Leicht": dict(building, C_spec=40.0),
    "Massiv": dict(building, C_spec=120.0),
}
selected_filter="lookahead"  # rolling_mean, exponential, ramp_rate oder lookahead
profiles={name:building_times(**v) for name,v in buildings.items()}
display(pd.DataFrame(profiles).loc[["tau","charging_time","discharging_time","capacity"]].rename(index={"tau":"Zeitkonstante [h]","charging_time":"Ladedauer [h]","discharging_time":"Entladedauer [h]","capacity":"Speicherkapazität [Wh/m² NGF]"}).round(2))
params_by_building={name:filter_params(v,advantage_threshold,mean_factor,filter_factor,horizon_factor) for name,v in profiles.items()}
signals_buildings={name:process(base,x,source,selected_filter,**v) for name,v in params_by_building.items()}
compare({"Ausgangssignal":base,**signals_buildings},interval,phase_cutoff)

### Auswertung C2 – hier antworten
1. Welcher Gebäudeparameter verändert sich? Wie beeinflusst er die Zeitkonstante, Lade- und Entladedauer?
2. Wie unterscheiden sich die daraus abgeleiteten Aktivierungssignale? Würde ein anderes Filterverfahren zu anderen Schlussfolgerungen führen?
3. Welche zusätzlichen Informationen wären erforderlich, um die tatsächliche Lastverschiebung, Kosten oder Emissionswirkung zu bewerten?
4. Formuliere eine überprüfbare Hypothese für PEExcel, z. B. zum Anteil des Heizstroms in günstigen Stunden, zum Komfort oder zu Lastspitzen.

**Antwort:** …

## 6. Export für die Gebäudesimulation

Exportiert wird ein kontinuierliches Aktivierungsprofil mit 8760 Stunden sowie die dokumentierte Zeitreihe und die Parameter. Eine mögliche theoretische Aktivierungsstrategie ist eine Sollwertverschiebung $T_{set}=T_{min}+s\Delta T$. In den vorliegenden PEExcel-Formeln wird das Signal dagegen auf das **elektrische flexible Leistungspotenzial** angewendet:

```excel
=INDEX(Signals;ZEILE()-ZEILE(Signals[#Kopfzeilen]);flex_Signals_selected_column)
```

```excel
=FLEX_is_used*FLEX_grid_maxpower_Wm2*[@Signal]
```

Die erste Formel liest den stündlichen Wert aus der per Kopfzeile gewählten Signalspalte. Die zweite skaliert das elektrische Netz-Flexleistungslimit in W/m². Ein Wert von 0,5 gibt damit 50 % des eingestellten Limits frei, nicht zwingend 50 % der tatsächlichen flexiblen Last und auch nicht 50 % des Komfortbands. Bei Signal 0 bleibt eine erforderliche Grundversorgung möglich; andere Flexquellen und die Regelungslogik sind gesondert zu berücksichtigen. Für den Import muss in der verwendeten Workbook-Version ein geeignetes Signalprofil in `Signals` verfügbar sein. Das endgültige Importformat ist noch zu prüfen. Die synthetische Modellzeitachse ist noch nicht zeitzonenrichtig mit PEExcel verifiziert.

In [ ]:
# EXPERIMENT – Ergebnis auswählen
selected_building="Massiv"
s=signals_buildings[selected_building]
export_signal(s,year,selected_building+"_"+selected_filter,
              dict(source=source,method=method,window=window,activation=activation,share=share,
                   threshold=threshold,filter=selected_filter,**buildings[selected_building],
                   **params_by_building[selected_building]),"signal_C")

### Schlussfolgerung und Abgabe
Dokumentiere Gebäudeparameter, Filterparameter, wichtigste Kennzahlen und deine Hypothese. Speichere das ausgeführte Notebook und das exportierte Signal. Vergleiche anschließend in PEExcel eine Referenz ohne zusätzliche Flexibilität, das Ausgangssignal aus Notebook 01 und das gebäudeangepasste Signal aus Notebook 02. Halte sonstige Eingaben konstant; prüfe Komfort, Strombedarf und Lastverlauf.

**Abgabe:** Ergänze die übernommenen IN-/OUT-Kennwerte deiner PEExcel-Variante, vergleiche berechnete und angezeigte Zeitkonstante sowie Entladedauer. Erkläre Abweichungen anhand der Eingaben und Modellannahmen. Beurteile anschließend die Signalvarianten anhand von äquivalenten Vollaktivierungsstunden, Phasendauern und dem zeitlichen Bezug zur externen Führungsgröße; in PEExcel zusätzlich die tatsächlich realisierte elektrische Flexibilität und den Komfort.

### Weiterführende Unterlagen

- [PEExcel-Wissensplattform](https://simonschaluppe.github.io/peexcel/) – Modell und Eingabedaten.
- [PEExcel-Schema](https://github.com/simonschaluppe/peexcel/tree/master/data/schemas/current) – IN, OUT und SIM. Die aktuellen OUT-Ergänzungen können vom bereits veröffentlichten Schema abweichen.
- LV-Unterlage „Sending the right signal – Synthese und Verwendung von Netzdienlichkeitssignalen“, insbesondere Folien zu Signalcharakteristik, Aktivierungsfunktionen, Gebäudezeitkonstante sowie Lade-/Entladedauer.


![Überleitung zur tatsächlichen PEExcel-Regelung](assets/signal_to_building_control.svg)

In SIM2 wird das Signal mit `INDEX(Signals;ZEILE()-ZEILE(Signals[#Kopfzeilen]);flex_Signals_selected_column)` gelesen. Die elektrische Flexgrenze ergibt sich aus `FLEX_is_used*FLEX_grid_maxpower_Wm2*[@Signal]`. Bei s=0 bleibt die Mindestversorgung grundsätzlich möglich; s=1 gibt die konfigurierte zusätzliche elektrische Leistung frei. Die tatsächlich genutzte Leistung ist weiter begrenzt.
